# Batched LLM Reranking Experiments (O(N log N) Pairwise Sort)

This notebook runs reranking on the Prime dataset queries. It utilizes the OpenAI Batch API natively. 
Instead of submitting N^2 combinations, we implement an asynchronous layered QuickSort algorithm. 
It requires multiple rounds of polling but reduces API calls to O(N log N) per query.

In [ ]:
import os
import re
import ast
import json
import time
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
import sys
sys.path.append('..') # Add root to path to import stark_qa

from stark_qa import load_skb

load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4.5-preview-2025-02-27"

# Load KB once for generating document strings
kb = load_skb('prime', download_processed=True)


## 1. Load the Data Dump & Precalculate Ranks

In [ ]:
DATA_DUMP_PATH = "../experiments/prime/LLM_SAVED_RESPONSES/full_data_dump.csv"
df = pd.read_csv(DATA_DUMP_PATH)

def extract_ground_truths(gt_val):
    if pd.isna(gt_val): return []
    if isinstance(gt_val, str):
        try: return ast.literal_eval(gt_val)
        except: return []
    return gt_val

def extract_answer_list(results_str):
    if pd.isna(results_str): return []
    if isinstance(results_str, str):
        try:
            res_dict = ast.literal_eval(results_str)
            return res_dict.get('answer_list', [])[:20]
        except: return []
    return []

df['ground_truths_list'] = df['ground_truths'].apply(extract_ground_truths)
df['top_20_nodes'] = df['results'].apply(extract_answer_list)
print(f"Loaded {len(df)} queries from {DATA_DUMP_PATH}")


## 2. Query Selector

In [ ]:
def get_target_queries(df, target_type="best_worst", num_bad=10, num_good=10, query_ids=None):
    def rank_of_first_gt(row):
        gt_set = set(row['ground_truths_list'])
        for i, pred in enumerate(row['top_20_nodes']):
            if pred in gt_set: return i + 1
        return 999
    
    df = df.copy()
    df['first_gt_rank'] = df.apply(rank_of_first_gt, axis=1)
    
    if target_type == "all": return df.to_dict('records')
    if target_type == "specific" and query_ids:
        return df[df['id'].isin(query_ids)].to_dict('records')

    bad_df = df[(df['first_gt_rank'] >= 5) & (df['first_gt_rank'] <= 20)].sort_values(by='first_gt_rank', ascending=False)
    good_df = df[df['first_gt_rank'] == 1]
    
    bad_q = bad_df.head(num_bad).to_dict('records')
    good_q = good_df.head(num_good).to_dict('records')
    
    print(f"Selected {len(bad_q)} bad queries, {len(good_q)} good queries.")
    return bad_q + good_q

test_queries_bw = get_target_queries(df, target_type="best_worst", num_bad=5, num_good=5)


## 3. Sort State Machine
Using QuickSort inherently modeled for O(N log N) async execution. Each round submits only comparing Pivot against other members for active unsorted subset chunks.

In [ ]:
class QuerySortState:
    def __init__(self, qid, query, docs):
        self.qid = qid
        self.query = query
        self.chunks = [{"type": "UNSORTED", "data": docs}]  # Start with everything unsorted
        
    def get_pending_comparisons(self):
        comps = []
        for chunk in self.chunks:
            if chunk["type"] == "UNSORTED":
                arr = chunk["data"]
                if len(arr) <= 1: continue
                pivot = arr[0]
                for x in arr[1:]:
                    comps.append((self.qid, self.query, pivot, x))
        return comps

    def apply_results(self, results_dict):
        new_chunks = []
        for chunk in self.chunks:
            if chunk["type"] == "SORTED":
                new_chunks.append(chunk)
            else:
                arr = chunk["data"]
                if len(arr) <= 1:
                    new_chunks.append({"type": "SORTED", "data": arr})
                    continue
                
                pivot = arr[0]
                left = []  # Better than pivot (winners go here to rank higher)
                right = [] # Worse or tie
                
                for x in arr[1:]:
                    winner = results_dict.get((self.qid, pivot, x))
                    # If x is chosen as the better doc, x wins and goes to the top (left)
                    if winner == x:
                        left.append(x)
                    else:
                        right.append(x)
                        
                if left: new_chunks.append({"type": "UNSORTED", "data": left})
                new_chunks.append({"type": "SORTED", "data": [pivot]})
                if right: new_chunks.append({"type": "UNSORTED", "data": right})
                    
        self.chunks = new_chunks
        
    def is_done(self):
        return all(c["type"] == "SORTED" for c in self.chunks)
        
    def get_sorted_list(self):
        res = []
        for c in self.chunks:
            res.extend(c["data"])
        return res


## 4. Batch API Executor
Runs rounds of Batch API polling iteratively.

In [ ]:
def submit_and_poll_batch(client, jsonl_path, batch_txt, interval=15):
    print(f"\n-- Uploading {jsonl_path.name}...")
    with jsonl_path.open("rb") as f:
        upload = client.files.create(file=f, purpose="batch")
    
    b = client.batches.create(input_file_id=upload.id, endpoint="/v1/chat/completions", completion_window="24h")
    batch_txt.write_text(b.id)
    
    print(f"-- Batch created: {b.id}. Polling...")
    terms = {"completed", "failed", "expired", "cancelled"}
    while True:
        curr = client.batches.retrieve(b.id)
        c = curr.request_counts
        done, tot, fail = (c.completed, c.total, c.failed) if c else (0,0,0)
        print(f"   [{time.strftime('%H:%M:%S')}] {curr.status} | {done}/{tot} (Failed {fail})", end='\r')
        if curr.status in terms: 
            print() 
            break
        time.sleep(interval)
        
    if curr.status != "completed": raise RuntimeError(curr.status)
    return curr.output_file_id

def map_winner_from_response(ans, node1_id, node2_id):
    try: ans = ans.replace("'", "").replace('"', "").strip() 
    except: pass
    
    if "A" in ans or str(node1_id) in ans: return node1_id
    if "B" in ans or str(node2_id) in ans: return node2_id
    return node1_id # default to pivot tie if invalid

def run_tournament_sort_experiment(queries):
    base = Path("../experiments/batch_reranking_sort")
    base.mkdir(parents=True, exist_ok=True)
    
    states = [QuerySortState(q['id'], q['query'], q['top_20_nodes']) for q in queries]
    round_id = 0
    
    while True:
        round_id += 1
        comps = []
        for s in states:
            comps.extend(s.get_pending_comparisons())
            
        if not comps:
            print("All queries fully sorted!")
            break
            
        print(f"\n>>> ROUND {round_id} - Processing {len(comps)} comparisons...")
        req_path = base / f"round_{round_id}_req.jsonl"
        res_path = base / f"round_{round_id}_res.jsonl"
        batch_txt = base / f"round_{round_id}.txt"
        
        with open(req_path, 'w', encoding='utf-8') as f:
            for qid, query, doc1, doc2 in comps:
                d1_info = kb.get_doc_info(doc1, add_rel=True, compact=True)
                d2_info = kb.get_doc_info(doc2, add_rel=True, compact=True)
                n1_type = kb.get_node_type_by_id(doc1)
                n2_type = kb.get_node_type_by_id(doc2)
                
                # Mirroring old pairwise logic exactly
                prompt = (
                    f"The following two elements consist of an ID number, a type and a corresponding descriptive text:\n \n"
                    f"{doc1}, {n1_type}, {d1_info}. \n"
                    f"{doc2}, {n2_type}, {d2_info}. \n\n"
                    f"Find out which of the elements satisfies the following query better: \n"
                    f"{query} \n"
                    f"Return ONLY the corresponding ID number which corresponds to the element that satisfies "
                    f"the given query best. Nothing else."
                )
                req = {
                    "custom_id": f"q_{qid}__p_{doc1}__x_{doc2}",
                    "method": "POST", "url": "/v1/chat/completions",
                    "body": {"model": MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": 0.0, "max_tokens": 10}
                }
                f.write(json.dumps(req) + "\n")
                
        out_id = submit_and_poll_batch(client, req_path, batch_txt)
        client.files.content(out_id).write_to_file(res_path)
        
        # Parse results
        results_dict = {}
        with open(res_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                cid = data["custom_id"] # q_X__p_Y__x_Z
                parts = cid.replace("q_", "").split("__")
                qid = int(parts[0])
                d1 = int(parts[1].replace("p_",""))
                d2 = int(parts[2].replace("x_",""))
                
                try:
                    ans = data["response"]["body"]["choices"][0]["message"]["content"]
                    winner = map_winner_from_response(ans, d1, d2)
                except:
                    winner = d1
                results_dict[(qid, d1, d2)] = winner
                
        # Apply Results to the State
        for s in states:
            s.apply_results(results_dict)
            
    # Wrap up outputs
    final_queries = []
    state_dict = {s.qid: s.get_sorted_list() for s in states}
    for q in queries:
        output = q.copy()
        output['reranked'] = state_dict[q['id']]
        final_queries.append(output)
        
    return final_queries


## 5. View Metrics

In [ ]:
def analyze_metrics(final_queries):
    metrics = []
    def mrr(lst, gts):
        for i, v in enumerate(lst): 
            if v in gts: return 1.0/(i+1)
        return 0.0
        
    for q in final_queries:
        gts = set(q['ground_truths_list'])
        orig, rrk = q['top_20_nodes'], q['reranked']
        
        metrics.append({
            'id': q['id'],
            'orig_mrr': mrr(orig, gts), 'new_mrr': mrr(rrk, gts),
            'orig_h1': 1.0 if orig and orig[0] in gts else 0.0,
            'new_h1': 1.0 if rrk and rrk[0] in gts else 0.0,
            'orig_h5': 1.0 if set(orig[:5]) & gts else 0.0,
            'new_h5': 1.0 if set(rrk[:5]) & gts else 0.0
        })
        
    df_m = pd.DataFrame(metrics)
    df_m['mrr_diff'] = df_m['new_mrr'] - df_m['orig_mrr']
    
    print("\n=== RERANKING RESULTS ===")
    print(f"MRR:      {df_m['orig_mrr'].mean():.3f} -> {df_m['new_mrr'].mean():.3f} (Δ {df_m['mrr_diff'].mean():+.3f})")
    print(f"Hit@1:    {df_m['orig_h1'].mean():.3f} -> {df_m['new_h1'].mean():.3f}")
    print(f"Hit@5:    {df_m['orig_h5'].mean():.3f} -> {df_m['new_h5'].mean():.3f}")
    
    return df_m

# RUN EXPERIMENT:
# outputs = run_tournament_sort_experiment(test_queries_bw)
# analysis_df = analyze_metrics(outputs)
# display(analysis_df)
